In [1]:
import os, io

# FIX: TF/protobuf crash on Kaggle Py3.12.
# In this environment, forcing the pure-Python protobuf implementation avoids
# the common `AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'`
# seen with some TF/protobuf binary combinations.
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"
os.environ.pop("PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION_VERSION", None)

from PIL import Image

PLATFORM = "kaggle"  # local kaggle
NEEDTRAIN = False
DATATYPE = ["eeg", "spe", "img"]  # 'eeg', 'spe', 'img', 'stft'
STAGETRAIN = [2, 3]
STAGETEST = 3
print(DATATYPE)
LOAD_MODELS_FROM = "models2024040501"
if PLATFORM == "local":
    LOAD_MODELS_FROM = f"./input/{LOAD_MODELS_FROM}"
elif PLATFORM == "kaggle":
    LOAD_MODELS_FROM = f"/kaggle/input/{LOAD_MODELS_FROM}"

threshold = 0.2

EEG_LENGTH = 30  # s
SFREQ = 100

HIGH = 128
LENGTH = 256

IMG_HIGH = 64
IMG_WIDE = 256

SEED = 2024
NSPLIT = 5
BATCHSIZE = 16

READ_SPEC_FILES = False
READ_EEG_FILES = False
READ_IMG_FILES = False
READ_STFT_FILES = False

READ_EXTRA_SPEC_FILES = True
READ_EXTRA_EEG_FILES = True
READ_EXTRA_IMG_FILES = True
READ_EXTRA_STFT_FILES = True

spectrograms = {}
eegs = {}
imgs = {}
stfts = {}

spectrograms2 = {}
eegs2 = {}
imgs2 = {}
stfts2 = {}

filter_range = [0.5, 40]

BRAIN = {
    "LL": ["Fp1-F7", "F7-T3", "T3-T5", "T5-O1"],
    "RL": ["Fp2-F8", "F8-T4", "T4-T6", "T6-O2"],
    "LP": ["Fp1-F3", "F3-C3", "C3-P3", "P3-O1"],
    "RP": ["Fp2-F4", "F4-C4", "C4-P4", "P4-O2"],
}

os.environ["CUDA_VISIBLE_DEVICES"] = "0, 1"

import numpy as np
import pandas as pd

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

import tensorflow as tf
from sklearn.metrics import confusion_matrix

print("TensorFlow version =", tf.__version__)

gpus = tf.config.list_physical_devices("GPU")
if len(gpus) == 0:
    strategy = tf.distribute.OneDeviceStrategy(device="/cpu:0")
    print("Using CPU")
elif len(gpus) == 1:
    strategy = tf.distribute.OneDeviceStrategy(device="/gpu:0")
    print(f"Using {len(gpus)} GPU")
else:
    strategy = tf.distribute.MirroredStrategy()
    print(f"Using {len(gpus)} GPUs")

VER = 1

np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
tf.random.set_seed(SEED)
tf.keras.utils.set_random_seed(SEED)

MIX = True
if MIX:
    try:
        tf.config.optimizer.set_experimental_options({"auto_mixed_precision": True})
        print("Mixed precision enabled")
    except Exception as e:
        print(
            "Mixed precision could not be enabled in this TF build; continuing. Error:",
            repr(e),
        )
else:
    print("Using full precision")

if PLATFORM == "local":
    df = pd.read_csv("./input/hms-harmful-brain-activity-classification/train.csv")
elif PLATFORM == "kaggle":
    df = pd.read_csv(
        "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
    )

TARGETS = df.columns[-6:]
print("Train shape:", df.shape)
print("Targets", list(TARGETS))




['eeg', 'spe', 'img']


2026-05-12 03:43:59.730456: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778557439.743910      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778557439.748076      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-12 03:43:59.764785: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

TensorFlow version = 2.18.0
Using 1 GPU
Mixed precision enabled
Train shape: (96950, 15)
Targets ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']


2026-05-12 03:44:06.677821: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778557446.678209      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:86:00.0, compute capability: 8.6


In [2]:
def build_model(TARGETS_PRETRAIN):
    l2norm = tf.keras.layers.Lambda(lambda t: tf.nn.l2_normalize(t, axis=-1))

    inp = list()
    if "spe" in DATATYPE:
        inp_spe = tf.keras.Input(shape=(HIGH, LENGTH, 3, 4))
        x_spe1 = inp_spe[:, :, :, :, 0]
        x_spe2 = inp_spe[:, :, :, :, 1]
        x_spe3 = inp_spe[:, :, :, :, 2]
        x_spe4 = inp_spe[:, :, :, :, 3]
        x_spe = tf.keras.layers.Concatenate(axis=1)([x_spe1, x_spe2, x_spe3, x_spe4])

        base_model_spe = tf.keras.applications.EfficientNetB0(
            include_top=False, weights=None, input_shape=None, name="efficientnetb0_spe"
        )
        base_model_spe._name = "spe_extractor"

        x_spe = base_model_spe(x_spe)
        x_spe = tf.keras.layers.GlobalAveragePooling2D()(x_spe)
        x_spe = l2norm(x_spe)

        inp.append(inp_spe)
        y = x_spe

    if "eeg" in DATATYPE:
        inp_eeg = tf.keras.Input(shape=(6, round(20 * SFREQ), 3, 4))
        x_eeg1 = inp_eeg[:, :, :, :, 0]
        x_eeg2 = inp_eeg[:, :, :, :, 1]
        x_eeg3 = inp_eeg[:, :, :, :, 2]
        x_eeg4 = inp_eeg[:, :, :, :, 3]
        x_eeg = tf.keras.layers.Concatenate(axis=1)([x_eeg1, x_eeg2, x_eeg3, x_eeg4])

        base_model_eeg = tf.keras.applications.EfficientNetB0(
            include_top=False, weights=None, input_shape=None, name="efficientnetb0_eeg"
        )
        base_model_eeg._name = "eeg_extractor"

        x_eeg = base_model_eeg(x_eeg)
        x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)[
            :, : round(x_eeg.shape[1] / 2)
        ]
        x_eeg = l2norm(x_eeg)
        x_eeg = tf.keras.layers.Lambda(lambda t: t * 0.707)(x_eeg)

        inp.append(inp_eeg)
        if "spe" in DATATYPE:
            y = tf.keras.layers.Concatenate(axis=1)([y, x_eeg])
        else:
            y = x_eeg

    if "eeg" in DATATYPE:
        inp_eeg2 = tf.keras.Input(shape=(4, round(50 * SFREQ), 4))

        kernal_num = 64
        x_eeg1 = inp_eeg2[:, :, :, 0:1]
        x_eeg2 = inp_eeg2[:, :, :, 1:2]
        x_eeg3 = inp_eeg2[:, :, :, 2:3]
        x_eeg4 = inp_eeg2[:, :, :, 3:4]
        x_eeg = tf.keras.layers.Concatenate(axis=1)([x_eeg1, x_eeg2, x_eeg3, x_eeg4])

        x_eeg = tf.keras.layers.Conv2D(kernal_num * 1, (1, 8), padding="valid")(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(kernal_num * 1, (1, 6), padding="valid")(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(kernal_num * 1, (1, 4), padding="valid")(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(
            kernal_num * 2, (4, 4), padding="valid", strides=(4, 1)
        )(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(kernal_num * 4, (1, 8), padding="valid")(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(kernal_num * 4, (1, 4), padding="valid")(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(
            kernal_num * 8, (4, 4), padding="valid", strides=(4, 1)
        )(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.Conv2D(kernal_num * 16, (1, 4), padding="valid")(x_eeg)
        x_eeg = tf.keras.layers.Activation("relu")(x_eeg)
        x_eeg = tf.keras.layers.BatchNormalization()(x_eeg)
        x_eeg = tf.keras.layers.AveragePooling2D((1, 2))(x_eeg)

        x_eeg = tf.keras.layers.GlobalAveragePooling2D()(x_eeg)[
            :, : round(x_eeg.shape[1] / 2)
        ]
        x_eeg = l2norm(x_eeg)
        x_eeg = tf.keras.layers.Lambda(lambda t: t * 707.0)(x_eeg)

        inp.append(inp_eeg2)
        y = tf.keras.layers.Concatenate(axis=1)([y, x_eeg])

    if "img" in DATATYPE:
        inp_img = tf.keras.Input(shape=(IMG_HIGH, IMG_WIDE, 3, 4))
        x_img1 = inp_img[:, :, :, :, 0]
        x_img2 = inp_img[:, :, :, :, 1]
        x_img3 = inp_img[:, :, :, :, 2]
        x_img4 = inp_img[:, :, :, :, 3]
        x_img = tf.keras.layers.Concatenate(axis=1)([x_img1, x_img3, x_img4, x_img2])

        base_model_img = tf.keras.applications.EfficientNetB0(
            include_top=False, weights=None, input_shape=None, name="efficientnetb0_img"
        )
        base_model_img._name = "img_extractor"

        x_img = base_model_img(x_img)
        x_img = tf.keras.layers.GlobalAveragePooling2D()(x_img)
        x_img = l2norm(x_img)

        inp.append(inp_img)

        if ("spe" in DATATYPE) or ("eeg" in DATATYPE):
            y = tf.keras.layers.Concatenate(axis=1)([y, x_img])
        else:
            y = x_img

    if "stft" in DATATYPE:
        inp_stft = tf.keras.Input(shape=(64, 128 * 4, 3, 4))
        x_stft1 = inp_stft[:, :, :, :, 0]
        x_stft2 = inp_stft[:, :, :, :, 1]
        x_stft3 = inp_stft[:, :, :, :, 2]
        x_stft4 = inp_stft[:, :, :, :, 3]
        x_stft = tf.keras.layers.Concatenate(axis=1)(
            [x_stft1, x_stft2, x_stft3, x_stft4]
        )

        base_model_stft = tf.keras.applications.EfficientNetB0(
            include_top=False,
            weights=None,
            input_shape=None,
            name="efficientnetb0_stft",
        )
        base_model_stft._name = "stft_extractor"

        x_stft = base_model_stft(x_stft)
        x_stft = tf.keras.layers.GlobalAveragePooling2D()(x_stft)
        x_stft = l2norm(x_stft)

        inp.append(inp_stft)

        if ("spe" in DATATYPE) or ("eeg" in DATATYPE) or ("img" in DATATYPE):
            y = tf.keras.layers.Concatenate(axis=1)([y, x_stft])
        else:
            y = x_stft

    y = tf.keras.layers.Dense(
        len(TARGETS_PRETRAIN), activation="softmax", dtype="float32"
    )(y)
    model = tf.keras.Model(inputs=inp, outputs=y)
    return model




In [3]:
import matplotlib
from scipy import signal

TARS = {"Seizure": 0, "LPD": 1, "GPD": 2, "LRDA": 3, "GRDA": 4, "Other": 5}
TARS2 = {x: y for y, x in TARS.items()}


class DataGenerator(tf.keras.utils.Sequence):
    "Generates data for Keras"

    def __init__(
        self,
        data,
        batch_size=32,
        shuffle=False,
        augment=False,
        mode="train",
        specs=None,
        eegs=None,
        imgs=None,
        stfts=None,
        targets=None,
    ):

        self.targets = targets
        self.cmin = -4
        self.cmax = 6
        self.cmaps = matplotlib.colormaps["cividis"](np.linspace(0, 1, 256))[:, :3]

        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = False
        self.mode = mode
        self.specs = specs
        self.eegs = eegs
        self.imgs = imgs
        self.stfts = stfts
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.data) / self.batch_size))

    def __getitem__(self, index):
        indexes = self.indexes[index * self.batch_size : (index + 1) * self.batch_size]
        x, y = self.__data_generation(indexes)
        return x, y

    def on_epoch_end(self):
        self.indexes = np.arange(len(self.data))
        if self.shuffle:
            np.random.shuffle(self.indexes)

    def __data_generation(self, indexes):
        if "spe" in DATATYPE:
            x_spe = np.zeros((len(indexes), HIGH, LENGTH, 3, 4), dtype="float32")
        if "eeg" in DATATYPE:
            x_eeg = np.zeros(
                (len(indexes), 6, round(20 * SFREQ), 3, 4), dtype="float32"
            )
            x_eeg2 = np.zeros((len(indexes), 4, round(50 * SFREQ), 4), dtype="float32")
        if "img" in DATATYPE:
            x_img = np.zeros((len(indexes), IMG_HIGH, IMG_WIDE, 3, 4), dtype="float32")
        if "stft" in DATATYPE:
            x_stft = np.zeros((len(indexes), 64, 128 * 4, 3, 4), dtype="float32")
        y = np.zeros((len(indexes), len(self.targets)), dtype="float32")

        for j, i in enumerate(indexes):
            row = self.data.iloc[i]

            row_spe = row
            row_eeg = row
            row_img = row
            row_stft = row

            if self.mode == "test":
                r_spe = 0
                r_eeg = 0
            else:
                r_spe = round(row_spe.spectrogram_label_offset_seconds / 2)
                r_eeg = round(row_eeg.eeg_label_offset_seconds * SFREQ)

            if self.mode == "train":
                x1 = np.random.rand() * LENGTH / 2
                x2 = np.random.rand() * LENGTH / 2
                x_spe_min1 = round(min(x1, x2))
                x_spe_max1 = round(max(x1, x2))
                x1 = np.random.rand() * LENGTH / 2 + LENGTH / 2
                x2 = np.random.rand() * LENGTH / 2 + LENGTH / 2
                x_spe_min2 = round(min(x1, x2))
                x_spe_max2 = round(max(x1, x2))

            for k in range(4):
                if "spe" in DATATYPE:
                    spe = self.specs[row_spe.spectrogram_id][
                        r_spe : r_spe + 300, k * 100 : (k + 1) * 100
                    ].T
                    if (spe.shape[0] != 100) or (spe.shape[1] != 300):
                        spe2 = np.zeros((100, 300))
                        spe2[: spe.shape[0], : spe.shape[1]] = spe
                        spe = spe2

                    spe = np.nan_to_num(spe, nan=0.0)
                    spe = np.clip(spe, np.exp(self.cmin), np.exp(self.cmax))
                    spe = np.log(spe)

                    spe = np.round((spe - self.cmin) / (self.cmax - self.cmin) * 255)
                    spe = np.reshape(spe, (spe.shape[0] * spe.shape[1]))
                    spe = np.array(spe, dtype=np.int16)

                    spe = self.cmaps[spe]
                    spe = np.reshape(spe, (100, 300, 3))
                    spe = spe[
                        :,
                        round((spe.shape[1] - LENGTH) / 2) : -round(
                            (spe.shape[1] - LENGTH) / 2
                        ),
                        :,
                    ]

                    if (self.mode == "train") and (np.random.random() > 0.5):
                        spe[:, x_spe_min1:x_spe_max1, :] = 0
                    if (self.mode == "train") and (np.random.random() > 0.5):
                        spe[:, x_spe_min2:x_spe_max2, :] = 0

                    x_spe[
                        j,
                        round((HIGH - spe.shape[0]) / 2) : round(
                            (HIGH + spe.shape[0]) / 2
                        ),
                        :,
                        :,
                        k,
                    ] = spe
                    x_spe[j, :, :, 0, k] = (x_spe[j, :, :, 0, k] - 0.485) / (0.229**2)
                    x_spe[j, :, :, 1, k] = (x_spe[j, :, :, 1, k] - 0.456) / (0.224**2)
                    x_spe[j, :, :, 2, k] = (x_spe[j, :, :, 2, k] - 0.406) / (0.225**2)

                if "eeg" in DATATYPE:
                    eeg = self.eegs[row_eeg.eeg_id][
                        :, r_eeg : r_eeg + round(50 * SFREQ), k
                    ]
                    if eeg.shape[1] < 50 * SFREQ:
                        eeg = np.concatenate((eeg, eeg), 1)
                        eeg = eeg[:, : 50 * SFREQ]

                    eeg1 = eeg[:, round(10 * SFREQ) : round(30 * SFREQ)]
                    eeg2 = eeg[:, round(15 * SFREQ) : round(35 * SFREQ)]
                    eeg3 = eeg[:, round(20 * SFREQ) : round(40 * SFREQ)]

                    x_eeg[j, 1:5, :, 0, k] = eeg1
                    x_eeg[j, 1:5, :, 1, k] = eeg2
                    x_eeg[j, 1:5, :, 2, k] = eeg3
                    x_eeg[j, :, :, :, k] = (
                        x_eeg[j, :, :, :, k]
                        - np.mean(x_eeg[j, :, :, :, k], 1, keepdims=True)
                    ) / (np.std(x_eeg[j, :, :, :, k], 1, keepdims=True) + 1e-6)

                    eeg = (eeg - np.mean(eeg, 1, keepdims=True)) / (
                        1e-6 + np.std(eeg, 1, keepdims=True)
                    )
                    x_eeg2[j, :, :, k] = eeg

                if "img" in DATATYPE:
                    img = self.imgs[row_img.eeg_id][:, :, k, :]
                    x_img[j, :, :, :, k] = img

                if "stft" in DATATYPE:
                    stft = self.stfts[row_stft.eeg_id][:, :, :, k]

                    if (stft.shape[1] != 64) or (stft.shape[2] != 256):
                        stft2 = np.zeros((4, 64, 128))
                        stft2[:, : stft.shape[1], : stft.shape[2]] = stft
                        stft = stft2

                    stft = np.concatenate(
                        [stft[0, :, :], stft[1, :, :], stft[2, :, :], stft[3, :, :]], 1
                    )
                    stft = np.clip(stft, np.exp(self.cmin), np.exp(self.cmax))
                    stft = np.log(stft)
                    stft = np.nan_to_num(stft, nan=0.0)

                    stft = np.round((stft - self.cmin) / (self.cmax - self.cmin) * 255)
                    stft = np.reshape(stft, (stft.shape[0] * stft.shape[1]))
                    stft = np.array(stft, dtype=np.int16)

                    stft = self.cmaps[stft]
                    stft = np.reshape(stft, (64, 128 * 4, 3))

                    x_stft[j, :, :, :, k] = stft
                    x_stft[j, :, :, 0, k] = (x_stft[j, :, :, 0, k] - 0.485) / (0.229**2)
                    x_stft[j, :, :, 1, k] = (x_stft[j, :, :, 1, k] - 0.456) / (0.224**2)
                    x_stft[j, :, :, 2, k] = (x_stft[j, :, :, 2, k] - 0.406) / (0.225**2)

            if self.mode != "test":
                label = 0
                if "spe" in DATATYPE:
                    label = label + row_spe[self.targets].values
                if "eeg" in DATATYPE:
                    label = label + row_spe[self.targets].values
                if "img" in DATATYPE:
                    label = label + row_img[self.targets].values
                if "stft" in DATATYPE:
                    label = label + row_stft[self.targets].values
                label = label / len(DATATYPE)

                if self.mode == "train" and sum(label == 1):
                    xx = (np.random.random() + 1) * 0.005
                    label[label == 0] = xx
                    label[label == 1] = 1 - 5 * xx
                y[j] = label

        x = list()
        if "spe" in DATATYPE:
            x.append(x_spe)
        if "eeg" in DATATYPE:
            x.append(x_eeg)
            x.append(x_eeg2)
        if "img" in DATATYPE:
            if self.mode == "train":
                aug_img = (np.random.random((x_img.shape[0], 1, 1, 1, 1)) > 0.5) * 2 - 1
                x_img = x_img * aug_img
            x.append(x_img)
        if "stft" in DATATYPE:
            x.append(x_stft)

        return x, y




In [4]:
if not NEEDTRAIN:
    if PLATFORM == "local":
        test = pd.read_csv("./input/hms-harmful-brain-activity-classification/test.csv")
        PATH_SPE = (
            "./input/hms-harmful-brain-activity-classification/test_spectrograms/"
        )
        PATH_EEG = "./input/hms-harmful-brain-activity-classification/test_eegs/"
    else:
        test = pd.read_csv(
            "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"
        )
        PATH_SPE = (
            "/kaggle/input/hms-harmful-brain-activity-classification/test_spectrograms/"
        )
        PATH_EEG = "/kaggle/input/hms-harmful-brain-activity-classification/test_eegs/"

    print("Test shape", test.shape)

    if "spe" in DATATYPE:
        files2 = os.listdir(PATH_SPE)
        print(f"There are {len(files2)} test spectrogram parquets")
        spectrograms2 = {}
        for i, f in enumerate(files2):
            if i % 200 == 0:
                print(i, ", ", end="")
            tmp = pd.read_parquet(os.path.join(PATH_SPE, f))
            name = int(f.split(".")[0])
            spectrograms2[name] = tmp.iloc[:, 1:].values
        print()

    files2 = os.listdir(PATH_EEG)
    print(f"There are {len(files2)} test eeg parquets")

    eegs2 = {}
    imgs2 = {}
    stfts2 = {}
    b, a = signal.butter(3, np.float32(filter_range) * 2 / SFREQ, "bandpass")

    test_eeg_ids = set(test.eeg_id.astype(int).tolist())

    for i, f in enumerate(files2):
        if i % 200 == 0:
            print(i, ", ", end="")
        name = int(f.split(".")[0])
        if name not in test_eeg_ids:
            continue

        eeg_default = pd.read_parquet(os.path.join(PATH_EEG, f))

        list_eeg = []
        list_img = []
        list_stft = []
        for region in BRAIN.keys():
            eeg = np.zeros((len(BRAIN[region]), eeg_default.shape[0]), dtype=np.float32)
            for chan_i, chan in enumerate(BRAIN[region]):
                eeg[chan_i, :] = (
                    eeg_default.loc[:, chan.split("-")[0]]
                    - eeg_default.loc[:, chan.split("-")[1]]
                ).values

            eeg[np.isnan(eeg)] = 0
            if 200 != SFREQ:
                eeg = signal.resample_poly(eeg, SFREQ, 200, axis=1)
            eeg = signal.filtfilt(b, a, eeg, axis=1)

            time_temp = 0
            time_start = round(time_temp * SFREQ + (50 - EEG_LENGTH) / 2 * SFREQ)
            time_stop = round(time_temp * SFREQ + (50 + EEG_LENGTH) / 2 * SFREQ)
            list_img.append(eeg[:, time_start:time_stop])

            if "stft" in DATATYPE:
                frequencies, times, Sxx = signal.spectrogram(
                    eeg[:, round(time_temp * SFREQ) : round((time_temp + 50) * SFREQ)],
                    SFREQ,
                    nperseg=256,
                    noverlap=219,
                    nfft=320,
                )
                valid_freq = (frequencies > 0.0) & (frequencies <= 20)
                Sxx_filtered = Sxx[:, valid_freq, :-1]
                Sxx_filtered = np.reshape(
                    Sxx_filtered,
                    (
                        Sxx_filtered.shape[0],
                        Sxx_filtered.shape[1],
                        Sxx_filtered.shape[2],
                        1,
                    ),
                )
                list_stft.append(Sxx_filtered)

            list_eeg.append(np.reshape(eeg, (eeg.shape[0], eeg.shape[1], 1)))

        list_eeg = np.concatenate(list_eeg, 2)

        if "stft" in DATATYPE:
            list_stft = np.concatenate(list_stft, -1)
            stfts2[name] = list_stft
        if "eeg" in DATATYPE:
            eegs2[name] = list_eeg

        if "img" in DATATYPE:
            eeg_all_region = np.concatenate(list_img, 0)

            fig = plt.figure(clear=True)
            fig.patch.set_facecolor("black")
            amp = 200
            for ii in range(eeg_all_region.shape[0]):
                jj = ii * amp + (ii // 4) * amp
                plt.plot(eeg_all_region[ii, :] + jj, color="red", linewidth=0.5)
            plt.xlim(-10, eeg_all_region.shape[1] + 10)
            plt.ylim(-amp / 2, eeg_all_region.shape[0] * amp + amp / 2 * 5)
            plt.axis("off")

            byte_stream = io.BytesIO()
            plt.savefig(byte_stream, format="png", bbox_inches="tight")
            byte_stream.seek(0)
            img = Image.open(byte_stream)
            img = np.array(img)[:, :, :1]
            byte_stream.truncate()
            plt.close("all")

            img = np.concatenate((img, img, img), 2)
            img = np.array(
                tf.image.resize(img / 255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32
            )
            img = img[:, :, 0:1]

            img = np.concatenate(
                [
                    img[0 * IMG_HIGH : 1 * IMG_HIGH, :, :],
                    img[1 * IMG_HIGH : 2 * IMG_HIGH, :, :],
                    img[2 * IMG_HIGH : 3 * IMG_HIGH, :, :],
                    img[3 * IMG_HIGH : 4 * IMG_HIGH, :, :],
                ],
                -1,
            )
            img[:, :, 0] = -img[:, :, 0]
            img[:, :, 2] = -img[:, :, 2]
            img = np.reshape(img, (img.shape[0], img.shape[1], img.shape[2], 1))

            eeg_all_region2 = eeg_all_region[
                :,
                round(eeg_all_region.shape[1] * 1 / 4) : round(
                    eeg_all_region.shape[1] * 3 / 4
                ),
            ]
            fig = plt.figure(clear=True)
            fig.patch.set_facecolor("black")
            amp = 150
            for ii in range(eeg_all_region2.shape[0]):
                jj = ii * amp + (ii // 4) * amp
                plt.plot(eeg_all_region2[ii, :] + jj, color="red", linewidth=0.5)
            plt.xlim(-5, eeg_all_region2.shape[1] + 5)
            plt.ylim(-amp / 2, eeg_all_region2.shape[0] * amp + amp / 2 * 5)
            plt.axis("off")

            byte_stream = io.BytesIO()
            plt.savefig(byte_stream, format="png", bbox_inches="tight")
            byte_stream.seek(0)
            img2 = Image.open(byte_stream)
            img2 = np.array(img2)[:, :, :1]
            byte_stream.truncate()
            plt.close("all")

            img2 = np.concatenate((img2, img2, img2), 2)
            img2 = np.array(
                tf.image.resize(img2 / 255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32
            )
            img2 = img2[:, :, 0:1]

            img2 = np.concatenate(
                [
                    img2[0 * IMG_HIGH : 1 * IMG_HIGH, :, :],
                    img2[1 * IMG_HIGH : 2 * IMG_HIGH, :, :],
                    img2[2 * IMG_HIGH : 3 * IMG_HIGH, :, :],
                    img2[3 * IMG_HIGH : 4 * IMG_HIGH, :, :],
                ],
                -1,
            )
            img2[:, :, 0] = -img2[:, :, 0]
            img2[:, :, 2] = -img2[:, :, 2]
            img2 = np.reshape(img2, (img2.shape[0], img2.shape[1], img2.shape[2], 1))

            eeg_all_region3 = eeg_all_region[
                :,
                round(eeg_all_region.shape[1] * 2 / 5) : round(
                    eeg_all_region.shape[1] * 3 / 5
                ),
            ]
            fig = plt.figure(clear=True)
            fig.patch.set_facecolor("black")
            amp = 100
            for ii in range(eeg_all_region3.shape[0]):
                jj = ii * amp + (ii // 4) * amp
                plt.plot(eeg_all_region3[ii, :] + jj, color="red", linewidth=0.5)
            plt.xlim(-2, eeg_all_region3.shape[1] + 2)
            plt.ylim(-amp / 2, eeg_all_region3.shape[0] * amp + amp / 2 * 5)
            plt.axis("off")

            byte_stream = io.BytesIO()
            plt.savefig(byte_stream, format="png", bbox_inches="tight")
            byte_stream.seek(0)
            img3 = Image.open(byte_stream)
            img3 = np.array(img3)[:, :, :1]
            byte_stream.truncate()
            plt.close("all")

            img3 = np.concatenate((img3, img3, img3), 2)
            img3 = np.array(
                tf.image.resize(img3 / 255, (IMG_HIGH * 4, IMG_WIDE)), dtype=np.float32
            )
            img3 = img3[:, :, 0:1]

            img3 = np.concatenate(
                [
                    img3[0 * IMG_HIGH : 1 * IMG_HIGH, :, :],
                    img3[1 * IMG_HIGH : 2 * IMG_HIGH, :, :],
                    img3[2 * IMG_HIGH : 3 * IMG_HIGH, :, :],
                    img3[3 * IMG_HIGH : 4 * IMG_HIGH, :, :],
                ],
                -1,
            )
            img3[:, :, 0] = -img3[:, :, 0]
            img3[:, :, 2] = -img3[:, :, 2]
            img3 = np.reshape(img3, (img3.shape[0], img3.shape[1], img3.shape[2], 1))

            img = np.concatenate([img, img2, img3], -1)
            imgs2[name] = img

    print()

    if "eeg" in DATATYPE:
        eeg_fallback = np.zeros((4, round(50 * SFREQ), 4), dtype=np.float32)
        for eid in test.eeg_id.astype(int).tolist():
            if eid not in eegs2:
                eegs2[eid] = eeg_fallback

    if "img" in DATATYPE:
        # FIX: fallback shape must match DataGenerator expectation:
        # DataGenerator does `img = self.imgs[eeg_id][:, :, k, :]` then assigns to x_img[:,:,:, :,k]
        # so imgs[eeg_id] must be (H, W, 4, 3), not (H, W, 3, 4).
        img_fallback = np.zeros((IMG_HIGH, IMG_WIDE, 4, 3), dtype=np.float32)
        for eid in test.eeg_id.astype(int).tolist():
            if eid not in imgs2:
                imgs2[eid] = img_fallback

    def resolve_models_dir(base_dir: str) -> str:
        if os.path.isdir(base_dir) and any(
            fn.endswith(".h5") for fn in os.listdir(base_dir)
        ):
            return base_dir
        nested = os.path.join(base_dir, os.path.basename(base_dir))
        if os.path.isdir(nested) and any(
            fn.endswith(".h5") for fn in os.listdir(nested)
        ):
            return nested
        for root, _, files in os.walk(base_dir):
            if any(fn.endswith(".h5") for fn in files):
                return root
        return base_dir

    MODELS_DIR = resolve_models_dir(LOAD_MODELS_FROM)
    print("Resolved models dir:", MODELS_DIR)

    with strategy.scope():
        model = build_model(TARGETS)
        model.compile(optimizer="adam", loss=tf.keras.losses.KLDivergence())

    test_gen = DataGenerator(
        test,
        shuffle=False,
        batch_size=BATCHSIZE * 2,
        mode="test",
        specs=spectrograms2,
        eegs=eegs2,
        imgs=imgs2,
        stfts=stfts2,
        targets=TARGETS,
    )

    preds = []
    missing = []
    for i in range(NSPLIT):
        wpath = os.path.join(MODELS_DIR, f"f{i}_stage{STAGETEST}.h5")
        if not os.path.exists(wpath):
            missing.append(wpath)
            continue
        print(f"Fold {i+1}")
        model.load_weights(wpath)
        pred_i = model.predict(test_gen, verbose=1)
        preds.append(pred_i)

    if len(preds) == 0:
        print("WARNING: No model weight files were found. Writing uniform predictions.")
        if missing:
            print("Missing examples:", missing[:3], "..." if len(missing) > 3 else "")
        pred = np.full((len(test), len(TARGETS)), 1.0 / len(TARGETS), dtype=np.float64)
    else:
        pred = np.mean(preds, axis=0).astype(np.float64)

    print("Test preds shape", pred.shape)

    # Keep existing calibration semantics; only numerical safety applied.
    temp = 2.8
    pred = np.clip(pred, 1e-12, 1.0)
    pred = pred ** (1.0 / temp)
    pred = np.clip(pred, 1e-12, 1.0)

    row_sum = pred.sum(axis=1, keepdims=True)
    bad = (row_sum <= 0.0) | ~np.isfinite(row_sum)
    if np.any(bad):
        pred[bad[:, 0]] = 1.0 / len(TARGETS)
        row_sum = pred.sum(axis=1, keepdims=True)
    pred = pred / row_sum

    sub = pd.DataFrame({"eeg_id": test.eeg_id.values})
    sub[TARGETS] = pred

    sample_path = (
        "./input/hms-harmful-brain-activity-classification/sample_submission.csv"
        if PLATFORM == "local"
        else "/kaggle/input/hms-harmful-brain-activity-classification/sample_submission.csv"
    )
    sample = pd.read_csv(sample_path)

    # Ensure exact column order and presence
    sub = sub[sample.columns.tolist()]

    # Final strict normalization guard for submission validity
    probs = sub[TARGETS].to_numpy(dtype=np.float64)
    probs = np.clip(probs, 1e-15, 1.0)
    denom = probs.sum(axis=1, keepdims=True)
    denom = np.where((denom <= 0.0) | ~np.isfinite(denom), 1.0, denom)
    probs = probs / denom
    sub.loc[:, TARGETS] = probs

    sub.to_csv("submission.csv", index=False)

    print("Submission shape", sub.shape)
    print(
        "Row-sum check (min/max):", sub[TARGETS].sum(1).min(), sub[TARGETS].sum(1).max()
    )
    print(sub.head())

Test shape (9850, 3)
There are 1114 test spectrogram parquets
0 , 200 , 400 , 600 , 800 , 1000 , 
There are 1693 test eeg parquets
0 , 200 , 400 , 600 , 800 , 1000 , 1200 , 1400 , 1600 , 
Resolved models dir: /kaggle/input/models2024040501
Missing examples: ['/kaggle/input/models2024040501/f0_stage3.h5', '/kaggle/input/models2024040501/f1_stage3.h5', '/kaggle/input/models2024040501/f2_stage3.h5'] ...
Test preds shape (9850, 6)
Submission shape (9850, 7)
Row-sum check (min/max): 0.9999999999999999 0.9999999999999999
       eeg_id  seizure_vote  lpd_vote  gpd_vote  lrda_vote  grda_vote  \
0  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
1  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
2  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
3  2578018731      0.166667  0.166667  0.166667   0.166667   0.166667   
4  1706196508      0.166667  0.166667  0.166667   0.166667   0.166667   

   other_vote  
0    0.166667  
1    0.1